# Data Augmentation for Alzheimer's Disease MRI Classification

This notebook implements the training-only data augmentation of the OTSU-GWWHO framework. Five methods (rotation, translation, adaptive gamma correction, scaling and affine transformation) generate varied MRI samples that reduce class imbalance and overfitting while keeping the anatomical content. Rotation is done without cropping so that no brain region is lost. The test images are never augmented.

## Prerequisites

- Python 3.10+
- Required libraries: `numpy`, `opencv-python`, `pandas`, `tqdm`
- Output of `image_preprocessing.ipynb` (`./outputs/02_splits/train.csv`)

## Workflow

1. Load the training split
2. Define the five augmentation methods
3. Generate augmented images per class (MD ×1, MOD ×10, ND ×1, VMD ×1 per method)
4. Keep a link (`source_id`) from each augmented image to its original
5. Save the augmented training set and its summary

In [ ]:
# ==========================================================
# IMPORTS & CONFIGURATION
# ==========================================================

import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

SPLIT_DIR = "./outputs/02_splits"
AUGMENTED_DIR = "./outputs/03_augmented"
CLASS_NAMES = ["MD", "MOD", "ND", "VMD"]

AUGMENTATION_METHODS = ["rotation", "translation", "gamma", "scaling", "affine"]
AUG_PER_METHOD = {"MD": 1, "MOD": 10, "ND": 1, "VMD": 1}   # augmented images per method

ROTATION_RANGE = 15      # degrees
SHIFT_RANGE = 0.10       # width / height shift
ZOOM_RANGE = 0.20        # scaling factor 0.8 - 1.2
SHEAR_RANGE = 0.20       # affine shear
GAMMA_JITTER = (0.85, 1.15)
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

## Rotation and Translation

Rotation by a random angle in [−15°, +15°] **without cropping** (the canvas is enlarged, then resized back), and random horizontal / vertical translation up to 10%.

In [ ]:
# ==========================================================
# ROTATION & TRANSLATION
# ==========================================================

def rotation(img):
    angle = rng.uniform(-ROTATION_RANGE, ROTATION_RANGE)
    h, w = img.shape
    m = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    cos, sin = abs(m[0, 0]), abs(m[0, 1])
    new_w, new_h = int(h * sin + w * cos), int(h * cos + w * sin)
    m[0, 2] += new_w / 2 - w / 2
    m[1, 2] += new_h / 2 - h / 2
    rotated = cv2.warpAffine(img, m, (new_w, new_h), borderValue=0)
    return cv2.resize(rotated, (w, h), interpolation=cv2.INTER_AREA)


def translation(img):
    h, w = img.shape
    tx = rng.uniform(-SHIFT_RANGE, SHIFT_RANGE) * w
    ty = rng.uniform(-SHIFT_RANGE, SHIFT_RANGE) * h
    m = np.float32([[1, 0, tx], [0, 1, ty]])
    return cv2.warpAffine(img, m, (w, h), borderValue=0)

## Adaptive Gamma Correction

Gamma is computed from the mean intensity of each image (γ = log 0.5 / log μ), so dark images are brightened and bright images are darkened. A small random jitter adds diversity.

In [ ]:
# ==========================================================
# ADAPTIVE GAMMA CORRECTION
# ==========================================================

def adaptive_gamma(img):
    norm = img.astype(np.float32) / 255.0
    fg = norm[norm > 0]
    mean = np.clip(fg.mean() if fg.size else norm.mean(), 1e-3, 1 - 1e-3)
    gamma = np.log(0.5) / np.log(mean) * rng.uniform(*GAMMA_JITTER)
    gamma = float(np.clip(gamma, 0.3, 3.0))
    return np.round(np.power(norm, gamma) * 255).astype(np.uint8)

## Scaling and Affine Transformation

Scaling with a random factor in [0.8, 1.2], and an affine transformation that combines rotation, translation, scaling and shearing.

In [ ]:
# ==========================================================
# SCALING & AFFINE
# ==========================================================

def scaling(img):
    h, w = img.shape
    s = rng.uniform(1 - ZOOM_RANGE, 1 + ZOOM_RANGE)
    m = cv2.getRotationMatrix2D((w / 2, h / 2), 0, s)
    return cv2.warpAffine(img, m, (w, h), borderValue=0)


def affine(img):
    h, w = img.shape
    angle = np.deg2rad(rng.uniform(-ROTATION_RANGE, ROTATION_RANGE))
    s = rng.uniform(1 - ZOOM_RANGE, 1 + ZOOM_RANGE)
    shear = rng.uniform(-SHEAR_RANGE, SHEAR_RANGE)
    tx = rng.uniform(-SHIFT_RANGE, SHIFT_RANGE) * w
    ty = rng.uniform(-SHIFT_RANGE, SHIFT_RANGE) * h
    rot = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
    a = s * rot @ np.array([[1, shear], [0, 1]])
    c = np.array([w / 2, h / 2])
    t = c - a @ c + np.array([tx, ty])
    m = np.hstack([a, t[:, None]]).astype(np.float32)
    return cv2.warpAffine(img, m, (w, h), borderValue=0)


AUGMENTERS = {"rotation": rotation, "translation": translation,
              "gamma": adaptive_gamma, "scaling": scaling, "affine": affine}

## Augment the Training Set

Applies the five methods to every training image. Each original keeps its `source_id`, so its augmented copies stay in the same cross-validation fold.

In [ ]:
# ==========================================================
# AUGMENT TRAINING SET
# ==========================================================

train_df = pd.read_csv(os.path.join(SPLIT_DIR, "train.csv"))
rows = []

for _, r in tqdm(train_df.iterrows(), total=len(train_df), desc="Augmenting"):
    rows.append({**r.to_dict(), "source_id": r["image_id"], "aug_method": "original", "is_augmented": 0})

    img = cv2.imread(r["path"], cv2.IMREAD_GRAYSCALE)
    out_dir = os.path.join(AUGMENTED_DIR, r["label"])
    os.makedirs(out_dir, exist_ok=True)
    stem = os.path.splitext(os.path.basename(r["path"]))[0]

    for method in AUGMENTATION_METHODS:
        for k in range(AUG_PER_METHOD[r["label"]]):
            out_path = os.path.join(out_dir, f"{stem}__{method}_{k}.png")
            cv2.imwrite(out_path, AUGMENTERS[method](img))
            rows.append({"image_id": f"{r['label']}/{stem}__{method}_{k}", "path": out_path,
                         "label": r["label"], "subject_id": r["subject_id"],
                         "source_id": r["image_id"], "aug_method": method, "is_augmented": 1})

aug_df = pd.DataFrame(rows)
aug_df.to_csv(os.path.join(AUGMENTED_DIR, "train_augmented.csv"), index=False)

## Augmentation Summary

Number of original, augmented and total training images per class.

In [ ]:
# ==========================================================
# SUMMARY
# ==========================================================

summary = pd.DataFrame({
    "Training Images": aug_df[aug_df.is_augmented == 0]["label"].value_counts(),
    "Augmented Images": aug_df[aug_df.is_augmented == 1]["label"].value_counts(),
}).reindex(CLASS_NAMES)
summary["Total Training Images"] = summary.sum(axis=1)
summary.loc["Total"] = summary.sum()
print(summary)